In [0]:
import json
import pandas as pd
import os

In [0]:
%pip install openpyxl

In [0]:
def read_from_config(relative_path: str):
    """
    Reads a JSON config file from the Git-based Databricks Repo.

    Parameters:
    - relative_path: str, path to config file relative to notebook

    Returns:
    - List of config entries
    """
    with open(relative_path, "r") as f:
        config_data = json.load(f)
    return config_data


In [0]:
import pandas as pd
from pyspark.sql import functions as F

def clean_column_names(df):
    """
    Cleans column names for Spark compatibility:
    - Strips leading/trailing spaces
    - Replaces spaces and invalid chars with underscores
    - Collapses multiple underscores into one
    - Removes trailing underscores
    - Converts to lowercase
    Works for both Pandas and Spark DataFrames.
    """
    if hasattr(df, "columns") and not hasattr(df, "withColumnRenamed"):  # Pandas
        new_cols = (
            pd.Series(df.columns)
            .str.strip()
            .str.replace(r"[ ,;{}()\n\t=-]", "_", regex=True)  # include hyphen
            .str.replace(r"__+", "_", regex=True)
            .str.rstrip("_")
            .str.lower()
            .tolist()
        )
        df.columns = new_cols
        return df

    elif hasattr(df, "withColumnRenamed"):  # Spark
        new_cols = [
            col_name.strip()
            .replace("-", "_")  # replace hyphen
            .replace(" ", "_")
            .replace(",", "_")
            .replace(";", "_")
            .replace("{", "_")
            .replace("}", "_")
            .replace("(", "_")
            .replace(")", "_")
            .replace("\n", "_")
            .replace("\t", "_")
            .replace("=", "_")
            for col_name in df.columns
        ]
        print(new_cols)
        new_cols = [c.lower() for c in new_cols]
        for old, new in zip(df.columns, new_cols):
            if old != new:
                df = df.withColumnRenamed(old, new)
        return df

    else:
        raise TypeError("Unsupported DataFrame type for column cleaning")



def enforce_string_for_object_columns(pdf: pd.DataFrame) -> pd.DataFrame:
    """
    Converts all object dtype columns in Pandas DataFrame to string
    to avoid Arrow conversion issues when creating Spark DataFrames.
    """
    for col in pdf.select_dtypes(include=["object"]).columns:
        pdf[col] = pdf[col].astype(str)
    return pdf

In [0]:
def create_managed_table_from_file(file_path: str, file_type: str, table_name: str):
    """
    Reads a file of given type, cleans column names, enforces datatypes,
    and creates a managed Spark table.
    """
    file_type = file_type.lower()

    if file_type == "csv":
        df = spark.read.option("header", True).option("inferSchema", True).csv(file_path)
        df = clean_column_names(df)
        df.display()

    elif file_type == "json":
        with open(file_path, "r") as f:
            data = json.load(f)   # assumes full JSON array in file

        pdf = pd.DataFrame(data)
        pdf = clean_column_names(pdf)
        pdf = enforce_string_for_object_columns(pdf)  # ✅ fix for Arrow
        df = spark.createDataFrame(pdf)

    elif file_type == "excel":
        pdf = pd.read_excel(file_path)
        pdf = clean_column_names(pdf)
        pdf = enforce_string_for_object_columns(pdf)  # ✅ fix for Arrow
        df = spark.createDataFrame(pdf)

    else:
        raise ValueError(f"Unsupported file type: {file_type}")

    # ✅ Create managed Delta table
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

    print(f"✅ Managed table `{table_name}` created from `{file_path}`")
    return df

In [0]:

config_entries = read_from_config("/Workspace/Users/soumyamukherjee42@gmail.com/personal_development/Config.json")

for entry in config_entries:
    file_path = entry["file_path"]
    file_type = entry["file_type"]
    table_name = entry["table_name"]

    print(f"\n📄 Processing {file_path} ({file_type}) -> {table_name}")
    create_managed_table_from_file(file_path, file_type, table_name)

print("\n🎉 All tables created successfully.")
